# 03 — Multi-Metric Model Comparison & Cross-Validation
**AutoDS Scientific Methodology Demonstration**

This notebook demonstrates candidate algorithm benchmarking using Stratified Cross-Validation on training data, followed by champion model selection.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backend.app.tools.preprocessor import prepare_train_test_split
from backend.app.tools.ml_trainer import train_and_evaluate_model, evaluate_locked_champion_on_holdout

df = pd.read_csv("data/raw/31513e04_winequality-red.csv", sep=None, engine="python")
X_train, X_test, y_train, y_test, prep = prepare_train_test_split(df, target_column="quality", problem_type="classification", test_size=0.2, random_state=42)
print("Data partitioned for model evaluation.")


Data partitioned for model evaluation.



## 1. Candidate Model Cross-Validation
We evaluate LightGBM, Random Forest, Logistic Regression, and Dummy Baseline across 3-fold Stratified CV.


In [2]:
candidates = ["LogisticRegression", "RandomForest", "Baseline"]
results = []

for model_name in candidates:
    res = train_and_evaluate_model(
        model_name=model_name,
        problem_type="classification",
        X_train=X_train,
        y_train=y_train,
        feature_names=prep.feature_names,
        cv_folds=3,
        user_goal="Predict wine quality",
        track_mlflow=False
    )
    cv_score = res["metrics"]["cv_mean"]
    cv_std = res["metrics"]["cv_std"]
    print(f"Model: {model_name:<20} | CV Mean: {cv_score:.4f} (+/- {cv_std:.4f}) | Fit Time: {res['train_time_sec']:.3f}s")
    results.append(res)


Model: LogisticRegression   | CV Mean: 0.8019 (+/- 0.0157) | Fit Time: 0.014s
Model: RandomForest         | CV Mean: 0.8001 (+/- 0.0095) | Fit Time: 0.082s
Model: Baseline             | CV Mean: 0.5000 (+/- 0.0000) | Fit Time: 0.000s



## 2. Champion Selection & Holdout Evaluation
The champion model is fitted on the complete training set and evaluated on the untouched holdout test set exactly once.


In [3]:
champion = max(results, key=lambda x: x["metrics"]["cv_mean"])
print(f"Selected Champion Model: {champion['model_name']} (CV: {champion['metrics']['cv_mean']:.4f})")

holdout_res = evaluate_locked_champion_on_holdout(
    champion_exp=champion,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    user_goal="Predict wine quality",
    track_mlflow=False
)

test_metrics = champion["metrics"]["test"]
print(f"Final Touchless Holdout Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Final Touchless Holdout Macro-AUC: {test_metrics['roc_auc']:.4f}")


Selected Champion Model: LogisticRegression (CV: 0.8019)
Final Touchless Holdout Accuracy: 0.5809
Final Touchless Holdout Macro-AUC: 0.7821

